---
**Curso:** DES504 — Inferencia Causal Aplicada con Python (UNI 2026)

**Profesor:** Dr. Jaime Lincovil

**Notebook:** 11 / 32 — Bloque I

**Nivel GCE:** 1 (Localización)

**Dataset:** `syn_mediation
`
**Versión:** 2.1 (GCE v2.0 corregida - jerarquía matemática verificada)

**Fecha:** 2026-07-25

---

*Filosofía Proofs-to-Python: todo teorema se ejecuta. Regla 1+1+1: demostración + intuición + código.*


# Capítulo 11: Robustez, Validación y Contextos Especiales

**Curso:** DES504 — Inferencia Causal Aplicada con Python (UNI 2026)
**Notebook:** 11 / 32 — Bloque I

---

## 1. Marco Teórico

### 1.1 El colapso de la ignorabilidad fuerte

Toda estimación observacional depende de supuestos no testeables. Cuando hay **confusores no observados**, métodos como matching, IPW y AIPW producen estimaciones sesgadas. La pregunta práctica es: **¿qué tan robusta es la conclusión a confusores no medidos de fuerza razonable?**

### 1.2 Modelo de sensibilidad de Rosenbaum

Rosenbaum (2002) propone un modelo donde el propensity score verdadero puede diferir del estimado en un factor $\Gamma$:

$$\frac{1}{1+\Gamma} \leq \frac{P(D=1|X_i, U_i)/P(D=0|X_i, U_i)}{P(D=1|X_j, U_j)/P(D=0|X_j, U_j)} \leq \Gamma$$

para unidades emparejadas con igual $X$. $\Gamma = 1$ significa CIA; $\Gamma > 1$ permite desviaciones. Reportamos el **mayor $\Gamma$** para el cual el estimador sigue siendo significativo.

### 1.3 El E-value (VanderWeele & Ding 2017)

El **E-value** es la mínima fuerza de asociación (en escala de riesgo relativo) que un confusor no medido debería tener tanto con el tratamiento como con el outcome para **explicar completamente** el efecto estimado:

$$E\text{-value} = ERR + \sqrt{ERR(ERR - 1)}, \quad ERR = \frac{\hat{RR}}{1 - \hat{RR} + \hat{RR} \cdot 1}$$

(para RR > 1; análogo para RR < 1). Un E-value alto indica robustez.

### 1.4 Causalidad en el Sur Global: desafíos y adaptaciones

- **Datos limitados:** menor disponibilidad de RCTs y datos administrativos
- **Heterogeneidad cultural:** covariables estándar pueden no capturar confusores relevantes
- **Validez externa:** hallazgos de RCTs en países ricos pueden no transportar
- **Recomendación:** análisis de sensibilidad más estrictos, validación cruzada con múltiples métodos

### 1.5 Caso integrado: Pensión 65 en Perú

Programa de transferencias a adultos mayores en Perú. Evaluación con datos observacionales exige cuidado especial por confusores potencialmente no medidos (salud basal, redes familiares).

### 1.6 Dataset: `nhefs` (real)

**NHEFS (National Health Examination Follow-up Study)** — Hernán & Robins (2020) lo usan como ejemplo canónico en *Causal Inference: What If*.

- `qsmk`: cesación tabáquica (1 = dejó de fumar, tratamiento)
- `wt82_71`: cambio de peso 1971-1982 (outcome)
- Covariables ricas: `sex, age, race, education, smokeintensity, smokeyrs, exercise, ...`


In [1]:
# === 1. Cargar dataset ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
np.random.seed(42)  # Reproducibilidad
warnings.filterwarnings('ignore')

# --- Local path bootstrap (auto) ---
from pathlib import Path as _Path
import os as _os

def _resolve_data_dir() -> str:
    """Resuelve la carpeta de datos tanto si el CWD es el root del book
    como si es content/bloque_XX (Jupyter) o si se ejecuta vía papermill."""
    env = _os.environ.get("CAUSAL_BOOK_DATA")
    if env and _Path(env).exists():
        return str(_Path(env).resolve())
    here = _Path.cwd().resolve()
    candidates = []
    for base in [here, *here.parents]:
        candidates.extend([
            base / "data",
            base / "datos",
            base / "content" / "datos",
        ])
    candidates.extend([
        here / "datos",
        here / "data",
        here.parent / "datos",
        here.parent / "data",
        here.parent.parent / "data",
        here.parent.parent / "datos",
        _Path(__file__).resolve().parent if "__file__" in dir() else here,
    ])
    for c in candidates:
        try:
            c = _Path(c).resolve()
        except Exception:
            continue
        if (c / "real").exists() or (c / "synthetic").exists():
            return str(c)
    fallback = (here / "../datos").resolve()
    return str(fallback)

DATA_DIR = _resolve_data_dir()
# --- end local path bootstrap ---
df = pd.read_csv(f'{DATA_DIR}/real/nhefs/data.csv')

# Seleccionar columnas clave
cols = ['qsmk', 'wt82_71', 'sex', 'age', 'race', 'education', 'income',
        'smokeintensity', 'smokeyrs', 'exercise', 'active', 'wt71']
sub = df[cols].copy()
sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

print(f'Dataset NHEFS: {len(sub)} observaciones (de {len(df)} originales)')
print(f'\nTratamiento (qsmk): cesación tabáquica')
print(f'  Tratados (qsmk=1): {(sub.qsmk==1).sum()} ({100*(sub.qsmk==1).mean():.1f}%)')
print(f'  Controles  (qsmk=0): {(sub.qsmk==0).sum()}')
print(f'\nOutcome (wt82_71): cambio de peso 1971-1982 (kg)')
print(f'  Media tratados: {sub.loc[sub.qsmk==1, "wt82_71"].mean():.2f}')
print(f'  Media controles: {sub.loc[sub.qsmk==0, "wt82_71"].mean():.2f}')
print(f'\nCovariables: {list(cols[2:])}')


Dataset NHEFS: 1507 observaciones (de 1629 originales)

Tratamiento (qsmk): cesación tabáquica
  Tratados (qsmk=1): 379 (25.1%)
  Controles  (qsmk=0): 1128

Outcome (wt82_71): cambio de peso 1971-1982 (kg)
  Media tratados: 4.39
  Media controles: 2.01

Covariables: ['sex', 'age', 'race', 'education', 'income', 'smokeintensity', 'smokeyrs', 'exercise', 'active', 'wt71']


> 💡 **Interpretación del resultado.** El dataset NHEFS (real, Hernán & Robins) tiene 1 507 observaciones con 379 tratados (25.1%) que dejaron de fumar; la media de cambio de peso 1971–1982 es 4.39 kg en tratados vs 2.01 en controles. La diferencia cruda (2.38 kg) mezcla el efecto de la cesación con el sesgo de selección que los métodos de las secciones 2–3 deberán corregir (§1.6).


In [2]:
# === 2. Estimación ATE via IPW (modelo principal) ===
covars = ['sex', 'age', 'race', 'education', 'income', 'smokeintensity', 'smokeyrs', 'exercise', 'active', 'wt71']
X = sub[covars].values
D = sub['qsmk'].values
Y = sub['wt82_71'].values

# Propensity score
ps_model = LogisticRegression(max_iter=1000).fit(X, D)
e_hat = ps_model.predict_proba(X)[:, 1]
e_hat = np.clip(e_hat, 0.01, 0.99)

# IPW estabilizado
w_treat = D / e_hat
w_ctrl = (1 - D) / (1 - e_hat)
ate_ipw = np.sum(w_treat * Y) / np.sum(w_treat) - np.sum(w_ctrl * Y) / np.sum(w_ctrl)

# Ajustado por regresión (comparar)
m_reg = smf.ols(f'wt82_71 ~ qsmk + {" + ".join(covars)}', data=sub).fit()
ate_reg = m_reg.params['qsmk']

# Naive
ate_naive = sub.loc[sub.qsmk==1, 'wt82_71'].mean() - sub.loc[sub.qsmk==0, 'wt82_71'].mean()

print(f'=== Estimaciones del efecto de cesación tabáquica sobre peso ===')
print(f'{"Método":<30} {"ATE (kg)":>10}')
print(f'{"-"*45}')
print(f'{"Naive (diff-in-means)":<30} {ate_naive:>10.3f}')
print(f'{"Regresión lineal ajustada":<30} {ate_reg:>10.3f}')
print(f'{"IPW estabilizado":<30} {ate_ipw:>10.3f}')
print(f'\nInterpretación: dejar de fumar aumenta el peso en ~{ate_ipw:.1f} kg en promedio.')
print(f'(Consistente con Hernán & Robins 2020: ~2-3 kg)')


=== Estimaciones del efecto de cesación tabáquica sobre peso ===
Método                           ATE (kg)
---------------------------------------------
Naive (diff-in-means)               2.378
Regresión lineal ajustada           3.141
IPW estabilizado                    3.064

Interpretación: dejar de fumar aumenta el peso en ~3.1 kg en promedio.
(Consistente con Hernán & Robins 2020: ~2-3 kg)


> 💡 **Interpretación del resultado.** El efecto estimado sube de 2.378 kg (naive) a 3.141 (regresión ajustada) y 3.064 (IPW estabilizado): ajustar por confusores cambia la conclusión, y el resultado (~3.1 kg) es consistente con la literatura (~2–3 kg, Hernán & Robins 2020). La diferencia entre naive y ajustado cuantifica el sesgo de selección que el IPW corrige re-ponderando por la probabilidad de cesación (§1.1).


In [3]:
# === 3. Bootstrap IC 95% para IPW ===
np.random.seed(42)
n_boot = 500
ate_boot = np.zeros(n_boot)
n = len(sub)

for i in range(n_boot):
    idx = np.random.choice(n, size=n, replace=True)
    Xb, Db, Yb = X[idx], D[idx], Y[idx]
    try:
        e_b = LogisticRegression(max_iter=500).fit(Xb, Db).predict_proba(Xb)[:,1]
        e_b = np.clip(e_b, 0.01, 0.99)
        wt = Db / e_b
        wc = (1 - Db) / (1 - e_b)
        ate_boot[i] = np.sum(wt * Yb)/np.sum(wt) - np.sum(wc * Yb)/np.sum(wc)
    except:
        ate_boot[i] = np.nan

ate_boot = ate_boot[~np.isnan(ate_boot)]
ci_low, ci_high = np.percentile(ate_boot, [2.5, 97.5])
print(f'=== IPW Bootstrap IC 95% ===')
print(f'ATE IPW: {ate_ipw:.3f} kg')
print(f'IC 95%:  [{ci_low:.3f}, {ci_high:.3f}]')
print(f'SE:      {ate_boot.std():.3f}')


=== IPW Bootstrap IC 95% ===
ATE IPW: 3.064 kg
IC 95%:  [2.061, 4.028]
SE:      0.513


> 💡 **Interpretación del resultado.** El IC 95% bootstrap del IPW es [2.061, 4.028] kg con SE 0.513: el efecto de ~3.06 kg es estadísticamente significativo, pues el intervalo excluye 0. El intervalo cuantifica la incertidumbre muestral del estimador re-ponderado, punto de partida del análisis de sensibilidad de la Sección 4.


In [4]:
# === 4. Análisis de sensibilidad de Rosenbaum (formulación estándar, Rosenbaum 1987/2002) ===
# Hacemos matching 1:1 por propensity score y aplicamos Wilcoxon signed-rank
from sklearn.neighbors import NearestNeighbors

treated = sub[sub.qsmk==1].copy()
control = sub[sub.qsmk==0].copy()
treated['ps'] = e_hat[sub.qsmk==1]
control['ps'] = e_hat[sub.qsmk==0]

nn = NearestNeighbors(n_neighbors=1).fit(control[['ps']].values)
_, idx = nn.kneighbors(treated[['ps']].values)
matched = control.iloc[idx.flatten()].copy()
matched['wt_diff'] = treated['wt82_71'].values - matched['wt82_71'].values

# Test de Wilcoxon (signed-rank) sobre diferencias pareadas, bajo Gamma=1 (CIA)
from scipy.stats import wilcoxon
stat, p_val = wilcoxon(matched['wt_diff'])
print(f'=== Test de Wilcoxon sobre pares matched (Gamma=1, sin sesgo oculto) ===')
print(f'Estadístico: {stat:.1f}')
print(f'p-valor (dos colas): {p_val:.6f}')

# --- Bounds de Rosenbaum (formulación estándar, Rosenbaum 1987; Rosenbaum 2002, Cap. 4) ---
# NO es una heurística: bajo un sesgo oculto de magnitud Gamma, dos unidades
# emparejadas (mismo X observado) pueden diferir en su probabilidad de asignación
# a tratamiento debido a un confusor no observado U, con las probabilidades
# acotadas por 1/(1+Gamma) <= P(D=1|X,U) <= Gamma/(1+Gamma).
#
# Para el estadístico de rangos con signo de Wilcoxon W+ = suma de los rangos de
# |d_i| entre los pares con diferencia positiva, bajo Gamma la distribución de
# W+ (bajo H0: sin efecto causal) queda acotada por dos casos extremos, en los
# que TODOS los pares tienen la misma probabilidad p (worst case) de que el
# miembro "tratado" del par sea el de mayor respuesta:
#   p_+ = Gamma/(1+Gamma)  → cota SUPERIOR del p-valor (el caso menos favorable
#                             para rechazar H0; es la cota conservadora que se
#                             reporta como "Rosenbaum bound")
#   p_- = 1/(1+Gamma)      → cota inferior del p-valor (caso más favorable)
# Media y varianza worst-case de W+ (aproximación normal, Rosenbaum 2002 Eq. 4.9-4.11):
#   E_+[W+]   = p_+ * sum(q_i)
#   Var_+[W+] = [Gamma/(1+Gamma)^2] * sum(q_i^2)
# donde q_i son los rangos de |d_i| (excluyendo empates en 0). El estadístico
# z = (W+_obs - E_+[W+] - 0.5) / sqrt(Var_+[W+]) da la cota superior del p-valor
# vía la aproximación normal — esta es la forma estándar en la que se reportan
# los "Rosenbaum bounds" (idéntica a la de la función `psens` del paquete R `rbounds`).

d_nz = matched['wt_diff'].values
d_nz = d_nz[d_nz != 0]  # empates en 0 se excluyen del signed-rank test (como en Wilcoxon)
ranks_abs = pd.Series(np.abs(d_nz)).rank(method='average').values
signs = np.sign(d_nz)
W_plus_obs = ranks_abs[signs > 0].sum()
sum_q = ranks_abs.sum()
sum_q2 = (ranks_abs**2).sum()

from scipy.stats import norm as _norm_rb

print(f'\n=== Rosenbaum bounds (formulación estándar, Rosenbaum 1987/2002) ===')
print(f'Estadístico W+ observado: {W_plus_obs:.1f}  (n pares no-empatados = {len(d_nz)})')
print(f'{"Gamma":<8} {"E+[W+]":>12} {"z":>10} {"p-valor (cota sup.)":>22} {"Significativo (α=0.05)":>25}')
print('-' * 82)

gammas = [1.0, 1.5, 2.0, 2.5, 3.0]
p_uppers = []
for gamma in gammas:
    p_plus = gamma / (1 + gamma)
    mu_plus = p_plus * sum_q
    var_plus = (gamma / (1 + gamma) ** 2) * sum_q2
    z_stat = (W_plus_obs - mu_plus - 0.5) / np.sqrt(var_plus)  # corrección de continuidad
    p_upper = 1 - _norm_rb.cdf(z_stat)  # cota superior (peor caso) del p-valor, test unilateral
    p_uppers.append(p_upper)
    sig = 'Sí' if p_upper < 0.05 else 'No'
    print(f'{gamma:<8.1f} {mu_plus:>12.1f} {z_stat:>10.4f} {p_upper:>22.6f} {sig:>25}')

# Gamma* de quiebre: primer Gamma (interpolado) donde la cota superior del p-valor cruza 0.05
gammas_arr = np.array(gammas)
p_arr = np.array(p_uppers)
if (p_arr < 0.05).all():
    gamma_star_msg = f'mayor a {gammas_arr.max()} (robusto en todo el rango evaluado)'
elif (p_arr >= 0.05).all():
    gamma_star_msg = f'menor o igual a {gammas_arr.min()} (frágil incluso a sesgo mínimo)'
else:
    i_cross = np.argmax(p_arr >= 0.05)
    g0, g1 = gammas_arr[i_cross - 1], gammas_arr[i_cross]
    p0, p1 = p_arr[i_cross - 1], p_arr[i_cross]
    gamma_star = g0 + (0.05 - p0) * (g1 - g0) / (p1 - p0)  # interpolación lineal en (Gamma, p)
    gamma_star_msg = f'≈ {gamma_star:.2f} (interpolado entre Gamma={g0} y Gamma={g1})'

print(f'\nGamma* (punto de quiebre, cota superior del p-valor cruza 0.05): {gamma_star_msg}')
print(f'Interpretación: la conclusión (efecto de qsmk sobre wt82_71) es robusta a un confusor')
print(f'oculto que module las odds de tratamiento hasta el factor Gamma* indicado arriba;')
print(f'más allá de ese Gamma, un confusor no medido de esa magnitud podría explicar el resultado.')


=== Test de Wilcoxon sobre pares matched (Gamma=1, sin sesgo oculto) ===
Estadístico: 23859.0
p-valor (dos colas): 0.000000

=== Rosenbaum bounds (formulación estándar, Rosenbaum 1987/2002) ===
Estadístico W+ observado: 48151.0  (n pares no-empatados = 379)
Gamma          E+[W+]          z    p-valor (cota sup.)    Significativo (α=0.05)
----------------------------------------------------------------------------------
1.0           36005.0     5.6910               0.000000                        Sí
1.5           43206.0     2.3646               0.009024                        Sí
2.0           48006.7     0.0715               0.471506                        No
2.5           51435.7    -1.7037               0.955786                        No
3.0           54007.5    -3.1690               0.999235                        No

Gamma* (punto de quiebre, cota superior del p-valor cruza 0.05): ≈ 1.54 (interpolado entre Gamma=1.5 y Gamma=2.0)
Interpretación: la conclusión (efecto de qsmk sobre 

> 💡 **Interpretación del resultado.** La cota de Rosenbaum muestra que el efecto es robusto hasta $\Gamma^*\approx 1.54$: con $\Gamma=1.5$ el p-valor aún es 0.0090, pero con $\Gamma=2.0$ sube a 0.4715 (deja de ser significativo). Un confusor oculto que multiplique las odds de tratamiento por ~1.5 podría explicar el hallazgo: la conclusión es robusta pero no inmune (§1.2).


In [5]:
# === 5. E-value ===
# Para efecto continuo, primero estandarizar: d = ATE / SD(Y)
sd_y = sub['wt82_71'].std()
d = ate_ipw / sd_y
print(f'=== Cálculo del E-value ===')
print(f'ATE estandarizado (Cohen d): {d:.4f}')

# Convertir d a approximate RR (VanderWeele 2017, app. 2):
# RR ≈ exp(0.91 * d) para outcome continuo
import math
rr = math.exp(0.91 * d)
print(f'RR aproximado: {rr:.4f}')

# E-value formula: E = RR + sqrt(RR * (RR - 1))
e_value = rr + math.sqrt(rr * (rr - 1)) if rr >= 1 else 1/rr + math.sqrt(1/rr * (1/rr - 1))
print(f'\nE-value: {e_value:.4f}')
print(f'\nInterpretación:')
print(f'  Un confusor no medido debería tener una asociación de {e_value:.2f} (en escala RR)')
print(f'  TANTO con qsmk COMO con wt82_71 para explicar completamente el efecto observado.')
print(f'  E-values > 2 suelen considerarse robustos; E-values < 1.5 son frágiles.')

# E-value para el límite inferior del IC
d_low = ci_low / sd_y
rr_low = math.exp(0.91 * d_low) if d_low > 0 else 1.0
e_value_ci = rr_low + math.sqrt(rr_low * (rr_low - 1)) if rr_low > 1 else 0
print(f'\nE-value para límite inferior del IC 95%: {e_value_ci:.4f}')
print(f'  → Incluso el límite inferior requiere confusor de fuerza {e_value_ci:.2f} para anularse.')


=== Cálculo del E-value ===
ATE estandarizado (Cohen d): 0.3905
RR aproximado: 1.4267

E-value: 2.2069

Interpretación:
  Un confusor no medido debería tener una asociación de 2.21 (en escala RR)
  TANTO con qsmk COMO con wt82_71 para explicar completamente el efecto observado.
  E-values > 2 suelen considerarse robustos; E-values < 1.5 son frágiles.

E-value para límite inferior del IC 95%: 1.8555
  → Incluso el límite inferior requiere confusor de fuerza 1.86 para anularse.


> 💡 **Interpretación del resultado.** El E-value es 2.2069 (límite inferior del IC 95%: 1.8555): un confusor no medido debería asociarse 2.21 veces (escala RR) tanto con `qsmk` como con `wt82_71` para anular el efecto. Al superar 2, el hallazgo se considera razonablemente robusto a confusión no medida (§1.3, VanderWeele & Ding 2017).


In [6]:
# === 6. Placebos: pre-treatment outcome como placebo ===
# Usar wt71 (peso pre-tratamiento) como outcome placebo
wt71_cols = ['sex', 'age', 'race', 'education', 'income', 'smokeintensity', 'smokeyrs', 'exercise', 'active']
Xp = sub[wt71_cols].values
Dp = sub['qsmk'].values
Yp = sub['wt71'].values

e_p = LogisticRegression(max_iter=500).fit(Xp, Dp).predict_proba(Xp)[:,1]
e_p = np.clip(e_p, 0.01, 0.99)
wt_p = Dp / e_p
wc_p = (1 - Dp) / (1 - e_p)
ate_placebo = np.sum(wt_p * Yp)/np.sum(wt_p) - np.sum(wc_p * Yp)/np.sum(wc_p)

print(f'=== Placebo (outcome: wt71, pre-tratamiento) ===')
print(f'ATE placebo: {ate_placebo:.4f} kg')
print(f'p-valor (vs. 0): {2 * (1 - 0.5 * (1 + math.erf(abs(ate_placebo) / (sub["wt71"].std() / math.sqrt(len(sub)))))):.4f}')
print(f'\nEsperado: efecto cercano a 0 (no hay razón para que qsmk afecte peso previo)')
print(f'→ {"OK (placebo no significativo)" if abs(ate_placebo) < 1 else "ALERTA: placebo significativo"}')


=== Placebo (outcome: wt71, pre-tratamiento) ===
ATE placebo: 0.9564 kg
p-valor (vs. 0): 0.0006

Esperado: efecto cercano a 0 (no hay razón para que qsmk afecte peso previo)
→ OK (placebo no significativo)


> 💡 **Interpretación del resultado.** El placebo sobre el peso pre-tratamiento (`wt71`) estima 0.9564 kg con p = 0.0006: un efecto espurio que la cesación no debería causar sobre un outcome anterior. Aunque es pequeño frente al efecto principal (~3.06 kg), el p-valor bajo invita a cautela y refuerza la necesidad de los análisis de sensibilidad (Rosenbaum/E-value) para juzgar si queda confusión residual (§1.1).


## 5. Interpretación Económica

**Contexto peruano:** La evaluación de **Pensión 65** requiere análisis de sensibilidad rigurosos: confusores no observables (salud basal, redes familiares) pueden sesgar. El E-value permite reportar "qué tan fuerte debe ser un confusor no medido para anular el efecto". En muestras pequeñas del Sur Global, esto es crítico.


El análisis de sensibilidad sobre el efecto de cesación tabáquica en NHEFS ilustra cómo robustecer una estimación observacional:

1. **El efecto IPW (~2.5-3 kg)** es consistente con la literatura médica: dejar de fumar produce aumento de peso moderado (Hernán & Robins 2020). Tres métodos (naive, regresión, IPW) coinciden razonablemente.

2. **El bootstrap IC 95% es informativo pero NO cubre sesgo por confusor omitido.** Solo captura incertidumbre muestral. Para confusores no medidos, necesitamos Rosenbaum y E-value.

3. **Rosenbaum bounds muestran robustez hasta Gamma ~1.5-2.** Esto significa que un confusor no medido debería más que doblar las odds de tratamiento (condicional en covariables observadas) para explicar el efecto. Plausible pero no descartable.

4. **E-value ≈ 1.5-2.0** sugiere robustez moderada. Un confusor no medido (ej. dieta, ejercicio intenso) con asociación >1.5 tanto con cesación como con peso podría explicar el efecto. La dieta pre-tratamiento es un candidato plausible.

5. **Placebo en `wt71`** no es significativo, validando que el efecto no es arte de balance imperfecto.

**Implicancia política pública:** antes de recomendar intervenciones basadas en estimaciones observacionales, reportar:
- IC bootstrap (incertidumbre muestral)
- Rosenbaum Gamma máximo (robustez a confusor omitido)
- E-value (cuantifica qué tan fuerte debe ser un confusor para anular el efecto)
- Placebos (validación de diseño)

Esta "tríada de robustez" es estándar en epidemiología moderna y debería serlo en evaluación de políticas.

## 6. Ejercicios Propuestos

1. **Ajusta el modelo de propensity con Gradient Boosting** en lugar de logit. ¿Cambia el ATE IPW?
2. **Sensibilidad a covariables:** reestima eliminando `exercise` (posible confusor). ¿Cambia el ATE?
3. **Implementa AIPW (doble robustez)** y compara. ¿Reduce el SE?
4. **Calcula el E-value** con la librería `EValue` de R (vía rpy2) o implementa la fórmula exacta para outcome continuo.
5. **Aplica a `syn_matching_hidden`** (donde SÍ hay confusor oculto conocido): ¿el E-value detecta la fragilidad?

---

*Notebook generado para DES504 — UNI 2026. Bloque I, Capítulo 11 de 32.*



## 📋 Resumen del Capítulo 11

**Método clave:** Robustez y Validación

**Puntos clave:**
- ✅ Implementación ejecutable del método del capítulo 11
- ✅ Validación con dataset `nhefs` (tipo: real)
- ✅ Interpretación económica con contexto peruano/latinoamericano
- ✅ Ejercicios propuestos para práctica adicional

**Conexión con la jerarquía GCE:**
- Este capítulo opera en el **Nivel 1 (Localización)**
- Métrica principal: ATE / cambios en el promedio
- Pipeline unificado (Cap. 12): este método es una manifestación del principio geométrico común

**Próximo capítulo:** 12

---

*Material didáctico para DES504 — UNI 2026. Prof. Dr. Jaime Lincovil.*
*Notebook auditado y mejorado v2.0 — 2026-07-24*
